# 04 - Blob Storage indexer and enrichment skills

## Required permissions

- **You:** **Search Service Contributor** + **Search Index Data Contributor** on the workshop Search service, and **Storage Blob Data Contributor** on the sample container. Sign in with `az login` in the deployment tenant.
- **Search managed identity:** **Storage Blob Data Reader** on the container and **Cognitive Services OpenAI User** or equivalent on the parent Foundry resource for embeddings. Bicep grants the broader **Cognitive Services User** role.
- **Resources:** an existing Azure AI Search service with semantic ranker enabled, a Blob container, and an Azure OpenAI embedding deployment. No tenant administrator role is required for this lab.

**Resource prefix:** set `WORKSHOP_PREFIX` in your `.env`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). Your index, data source, skillset and indexer use `alice01-blob-*`; uploaded blobs use `alice01/blob/`. Fields, skills and document IDs are scoped inside your objects. Prefixes prevent naming collisions, not enforce access isolation.

Build the pipeline explicitly: **Blob Storage -> data source -> indexer -> Text Split + embedding + Shaper skills -> chunk/vector index -> keyword, vector, hybrid and semantic-ranked search**. Nothing uploads documents directly into Search: the indexer reads the blobs and executes the skillset. Notebook 06 builds a separate index using client-side chunking, embeddings and push ingestion.

Text Split creates overlapping text chunks; Azure OpenAI generates one embedding per chunk; Shaper preserves provenance. Text Split and Shaper are non-billable utility skills, but embeddings and semantic ranking have their own usage/quota. This lab reuses the deployed `workshop-embedding` deployment (`text-embedding-3-small`, 1,536 dimensions). No OCR or Vision skill is used; scanned PDFs need separate extraction.

Use the Search and Storage resources configured in `.env.azure`. Your user needs Search Service Contributor and Search Index Data Contributor on Search, and Storage Blob Data Contributor on the container. Search's system-assigned managed identity needs Storage Blob Data Reader on that container. The repository's [deployment guide](../infra/README.md) is one way to provision these resources and role assignments; if using existing resources, verify the permissions yourself. User credentials upload the source files; Search's identity reads them.

Search's identity also needs embedding inference on the parent Foundry resource: Cognitive Services OpenAI User is sufficient. The repository's Bicep grants the broader Cognitive Services User role, which includes this access. No model API key is used. Semantic ranker must be enabled on Search; confirm regional availability and quota before running the lab.

## 1. Configure your lab

Run `az login --tenant <AZURE_TENANT_ID>` in PowerShell and select your Python kernel. Run the kernel from the `python-notebooks` directory, just like labs 01-03. Settings load from `..\.env`, then `..\.env.azure`; file values override existing environment variables, and `.env.azure` wins for duplicate settings. Set `WORKSHOP_PREFIX` in your untracked `.env`. Rerun Configuration after changing settings.

Configure `AZURE_TENANT_ID`, `AZURE_SEARCH_ENDPOINT`, `AZURE_STORAGE_RESOURCE_ID`, `AZURE_STORAGE_BLOB_ENDPOINT`, `AZURE_STORAGE_CONTAINER_NAME`, `AZURE_AI_ACCOUNT_NAME` and `AZURE_AI_EMBEDDING_DEPLOYMENT_NAME` for your existing resources. The default embedding model is `text-embedding-3-small` with 1,536 dimensions; the model, skill and index must agree. Sample documents are read from `..\data\search`.

Every REST request is visible. Token providers refresh credentials without hiding the request, URL, headers or JSON payload.

In [ ]:
import json
import os
import re
import time
from datetime import datetime, timezone
from email.utils import format_datetime
from pathlib import Path
from urllib.parse import quote

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

load_dotenv(r'..\.env', override=True)
load_dotenv(r'..\.env.azure', override=True)
root = Path('..')
required = ('AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'AZURE_STORAGE_RESOURCE_ID',
            'AZURE_STORAGE_BLOB_ENDPOINT', 'AZURE_STORAGE_CONTAINER_NAME',
            'AZURE_AI_ACCOUNT_NAME', 'AZURE_AI_EMBEDDING_DEPLOYMENT_NAME', 'WORKSHOP_PREFIX')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. Configure .env and .env.azure for your resources.')
prefix = os.environ['WORKSHOP_PREFIX'].strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')
search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
storage_endpoint = os.environ['AZURE_STORAGE_BLOB_ENDPOINT'].rstrip('/')
storage_id = os.environ['AZURE_STORAGE_RESOURCE_ID']
container_name = os.environ['AZURE_STORAGE_CONTAINER_NAME']
embedding_endpoint = f"https://{os.environ['AZURE_AI_ACCOUNT_NAME']}.openai.azure.com"
embedding_deployment = os.environ['AZURE_AI_EMBEDDING_DEPLOYMENT_NAME']
embedding_model = os.getenv('AZURE_AI_EMBEDDING_MODEL_NAME', 'text-embedding-3-small')
embedding_dimensions = 1536
semantic_configuration = 'chunk-semantic'
api_version = '2025-09-01'
index_name = f'{prefix}-blob-chunks'
data_source_name = f'{prefix}-blob-source'
skillset_name = f'{prefix}-blob-skills'
indexer_name = f'{prefix}-blob-indexer'
blob_prefix = f'{prefix}/blob/'
credential = AzureCliCredential(tenant_id=os.environ['AZURE_TENANT_ID'])
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
storage_token = get_bearer_token_provider(credential, 'https://storage.azure.com/.default')

print('Search:', search_endpoint)
print('Index:', index_name)
print('Source folder:', f'{container_name}/{blob_prefix}')


### Check REST responses before continuing

The requests and payloads stay in their lab cells. This small check prints the HTTP error and response body before raising an exception to stop that cell; a failed request is never reported as ready. It also rejects unexpected successful statuses rather than relying only on `raise_for_status()`.

Blob uploads expect `201`. Search create/update calls accept `200`, `201` or `204`; indexer run requests expect `202`; status and search requests expect `200`. The initial indexer existence check also accepts `404` to mean it has not been created yet. Diagnostics do not print authorization headers; response bodies are limited to 8,000 characters. See [Search HTTP status codes](https://learn.microsoft.com/rest/api/searchservice/http-status-codes).

In [ ]:
from requests import HTTPError

def check_response(response, expected_statuses):
    if response.status_code in expected_statuses:
        return
    message = f'HTTP {response.status_code} {response.reason}; expected {expected_statuses}.'
    print(message)
    print('Response body:', response.text[:8000] or '(empty)')
    for header in ('x-ms-request-id', 'request-id', 'retry-after'):
        if response.headers.get(header):
            print(f'{header}: {response.headers[header]}')
    raise HTTPError(message, response=response, request=response.request)


## 2. Upload source documents to Blob Storage

The repository includes two original synthetic text documents. Upload only these files to your prefix; repeated uploads overwrite only the same two blobs. Add your own text-based PDF, DOCX, PPTX, HTML or TXT files to the same folder later using Storage Explorer. This lab's parser extracts text, not image content.

The storage endpoint remains private to authenticated callers (not anonymously readable). No account key or SAS is used.

In [ ]:
sample_files = sorted((root / 'data' / 'search').glob('*.txt'))
if len(sample_files) != 2:
    raise RuntimeError('Expected the two workshop sample documents under data/search.')
for path in sample_files:
    blob_name = f'{blob_prefix}{path.name}'
    response = requests.put(
        f'{storage_endpoint}/{container_name}/{quote(blob_name, safe="/")}',
        headers={
            'Authorization': f'Bearer {storage_token()}',
            'x-ms-version': '2023-11-03',
            'x-ms-date': format_datetime(datetime.now(timezone.utc), usegmt=True),
            'x-ms-blob-type': 'BlockBlob',
            'Content-Type': 'text/plain; charset=utf-8',
        },
        data=path.read_bytes(), timeout=(10, 90),
    )
    check_response(response, (201,))
    print('Uploaded:', blob_name)


## 3. Create the Blob data source

`ResourceId=...;` tells Search to authenticate to Storage using its system-assigned managed identity. `container.query` scopes the indexer to your virtual folder, so it does not ingest everyone else's workshop documents. This is a data selection rule, not a security boundary.

We deliberately omit a deletion policy in this introductory lab. Deleting a source blob alone does not promise removal of its indexed chunks. Configure and test a supported deletion-detection policy before using this pattern for content that must be removed automatically.

In [ ]:
# Register where the indexer will read documents; this does not ingest them yet.
data_source = {
    'name': data_source_name,
    'type': 'azureblob',
    # Search uses its managed identity to read Storage, not an account key or SAS.
    'credentials': {'connectionString': f'ResourceId={storage_id};'},
    # Read only this prefix within the container; this is not an access-control boundary.
    'container': {'name': container_name, 'query': blob_prefix},
}
# Your Entra token authorizes creating or updating this named Search data source.
response = requests.put(
    f'{search_endpoint}/datasources/{data_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=data_source, timeout=(10, 90),
)
# Accept the create/update success statuses, including an empty 204 response.
# Otherwise print the HTTP error/body and stop before reporting success.
check_response(response, (200, 201, 204))
print('Data source ready:', data_source_name)


## 4. Define a chunk index

Each chunk is a Search document. Index projections generate `chunk_id` and `parent_id`; do not manually map either. The complex `source` field holds Shaper provenance, and `title` holds the source filename.

`text_vector` stores 1,536 floats. Its HNSW profile uses cosine similarity and an Azure OpenAI vectorizer for query-time text embeddings. The skill and vectorizer must use the same deployment/model, and the skill dimensions must match this field. The `semantic` configuration prioritizes `title` and `chunk` for reranking. All three are visible in this REST payload.

If upgrading the old keyword-only lab, rerun from the top: reuploading the samples makes the indexer process them again. Adding a vector field alone does not backfill unchanged documents. Alternatively reset and run the indexer in section 8. Changing an existing vector field's dimensions can require recreating your own index; this notebook never deletes it automatically.

### Use an existing index as a starting point

You can inspect indexes in Azure AI Search and use their definitions as a baseline for your own. Open your Search service, select **Indexes**, then open an index. The **Fields** tab shows field types, searchable/filterable settings, complex fields and vector dimensions.

<img src="../docs/ai-search-index-fields.png" alt="Azure AI Search index Fields tab showing chunk fields, provenance and a 1536-dimensional vector" width="900">

Select **Edit JSON** to inspect the complete index definition, including fields, vector profiles and semantic configurations. This helps connect the portal settings to the REST payload in the next cell.

<img src="../docs/ai-search-index-json-editor.png" alt="Azure AI Search index JSON editor showing the index name and field definitions" width="700">

**Adapt the baseline, don't overwrite the original:** use your own `index_name`, remove response metadata such as `@odata.etag`, and review field attributes rather than copying every setting unchanged. Update embedding resource/deployment references, vector dimensions and semantic settings for your resources. The screenshots are illustrative; their names and field settings can differ from this notebook's payload. An index definition creates the schema, not the documents; the indexer populates it later.

In [ ]:
index = {
    'name': index_name,
    'fields': [
        {'name': 'chunk_id', 'type': 'Edm.String', 'key': True,
         'searchable': True, 'analyzer': 'keyword'},
        {'name': 'parent_id', 'type': 'Edm.String', 'filterable': True},
        {'name': 'chunk', 'type': 'Edm.String', 'searchable': True},
        {'name': 'title', 'type': 'Edm.String', 'searchable': True},
        {'name': 'text_vector', 'type': 'Collection(Edm.Single)',
         'searchable': True, 'retrievable': True, 'dimensions': embedding_dimensions,
         'vectorSearchProfile': 'chunk-vector-profile'},
        {'name': 'source', 'type': 'Edm.ComplexType', 'fields': [
            {'name': 'file_name', 'type': 'Edm.String', 'searchable': True},
            {'name': 'url', 'type': 'Edm.String'},
        ]},
    ],
    'vectorSearch': {
        'algorithms': [{'name': 'chunk-hnsw', 'kind': 'hnsw',
                        'hnswParameters': {'metric': 'cosine'}}],
        'profiles': [{'name': 'chunk-vector-profile', 'algorithm': 'chunk-hnsw',
                      'vectorizer': 'chunk-vectorizer'}],
        'vectorizers': [{
            'name': 'chunk-vectorizer', 'kind': 'azureOpenAI',
            'azureOpenAIParameters': {
                'resourceUri': embedding_endpoint, 'deploymentId': embedding_deployment,
                'modelName': embedding_model,
            },
        }],
    },
    'semantic': {
        'defaultConfiguration': semantic_configuration,
        'configurations': [{
            'name': semantic_configuration,
            'prioritizedFields': {
                'titleField': {'fieldName': 'title'},
                'prioritizedContentFields': [{'fieldName': 'chunk'}],
            },
        }],
    },
}
response = requests.put(
    f'{search_endpoint}/indexes/{index_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=index, timeout=(10, 90),
)
check_response(response, (200, 201, 204))
print('Index ready:', index_name)


## 5. Define chunking, embedding and provenance skills

The indexer cracks each source file into `/document/content` and metadata. Text Split writes `/document/chunks/*`. The embedding skill runs on each chunk and writes `/document/chunks/*/text_vector`. Shaper writes `/document/provenance`. Index projections send the text, vector, filename and provenance to the chunk index.

**PDF support:** this Blob indexer handles text-based PDFs as well as the sample TXT files. It extracts the PDF's embedded text, then the same split, embedding and projection pipeline indexes it. Upload PDFs to your `<prefix>/blob/` folder and request another indexer run after any current run finishes.

**Scanned or image-only PDFs need OCR**, which this lab does not configure. These skills also do not preserve physical PDF page numbers. See [notebook 05](05-blob-content-understanding.ipynb) for Content Understanding extraction with page metadata.

Omitting `apiKey` and `authIdentity` from both the embedding skill and vectorizer selects Search's system-assigned managed identity. An additional `cognitiveServices` skillset attachment is not required for this embedding skill. Embedding calls incur token charges.

**Text Split has two `textSplitMode` options:**

| Mode | How text is split |
| --- | --- |
| `pages` | Size-based chunks with configurable maximum length and overlap. The skill attempts to preserve sentence boundaries; these are **not physical PDF pages**. |
| `sentences` | Individual sentences, using sentence-ending punctuation and language rules. |

This lab uses `pages` with up to **1,000 characters** per chunk and **200-character overlap**. One PDF page can produce several chunks, and a chunk can span PDF pages. These settings make boundaries easy to inspect, but do not provide page-number citations.

Character-based versus token-based chunk length is a separate `unit` setting (`characters` or `azureOpenAITokens`), not another split mode. This payload omits `unit`, so the default is characters. See the [Text Split skill parameters](https://learn.microsoft.com/azure/search/cognitive-search-skill-textsplit#skill-parameters).

In [ ]:
skillset = {
    'name': skillset_name,
    'description': 'Split and embed blob text; preserve structured source provenance.',
    'skills': [
        {
            '@odata.type': '#Microsoft.Skills.Text.SplitSkill',
            'name': 'split-text', 'context': '/document',
            'textSplitMode': 'pages', 'maximumPageLength': 1000, 'pageOverlapLength': 200,
            'inputs': [{'name': 'text', 'source': '/document/content'}],
            'outputs': [{'name': 'textItems', 'targetName': 'chunks'}],
        },
        {
            '@odata.type': '#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill',
            'name': 'embed-chunks', 'context': '/document/chunks/*',
            'resourceUri': embedding_endpoint, 'deploymentId': embedding_deployment,
            'modelName': embedding_model, 'dimensions': embedding_dimensions,
            'inputs': [{'name': 'text', 'source': '/document/chunks/*'}],
            'outputs': [{'name': 'embedding', 'targetName': 'text_vector'}],
        },
        {
            '@odata.type': '#Microsoft.Skills.Util.ShaperSkill',
            'name': 'shape-provenance', 'context': '/document',
            'inputs': [
                {'name': 'file_name', 'source': '/document/metadata_storage_name'},
                {'name': 'url', 'source': '/document/metadata_storage_path'},
            ],
            'outputs': [{'name': 'output', 'targetName': 'provenance'}],
        },
    ],
    'indexProjections': {
        'selectors': [{
            'targetIndexName': index_name, 'parentKeyFieldName': 'parent_id',
            'sourceContext': '/document/chunks/*',
            'mappings': [
                {'name': 'chunk', 'source': '/document/chunks/*'},
                {'name': 'text_vector', 'source': '/document/chunks/*/text_vector'},
                {'name': 'title', 'source': '/document/metadata_storage_name'},
                {'name': 'source', 'source': '/document/provenance'},
            ],
        }],
        'parameters': {'projectionMode': 'skipIndexingParentDocuments'},
    },
}
response = requests.put(
    f'{search_endpoint}/skillsets/{skillset_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=skillset, timeout=(10, 90),
)
check_response(response, (200, 201, 204))
print('Skillset ready:', skillset_name)


## 6. Create the indexer

The indexer connects the data source, skillset and target index. Creating it starts its first run automatically. When this cell updates an existing idle indexer, it explicitly requests a new run; an already-running indexer must finish first. The first run can take several minutes while permissions propagate. No schedule is configured, so additional runs are explicit.

`maxFailedItems=0` and `maxFailedItemsPerBatch=0` prevent a partially failed run from being presented as a successful workshop outcome.

In [ ]:
response = requests.get(
    f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
)
check_response(response, (200, 404))
indexer_exists = response.status_code != 404
previous_run_start = None
if indexer_exists:
    previous_run = response.json().get('lastResult') or {}
    if previous_run.get('status') == 'inProgress':
        raise RuntimeError('The indexer is already running. Wait before updating it.')
    previous_run_start = previous_run.get('startTime')
indexer = {
    'name': indexer_name, 'dataSourceName': data_source_name,
    'targetIndexName': index_name, 'skillsetName': skillset_name,
    'parameters': {
        'maxFailedItems': 0, 'maxFailedItemsPerBatch': 0,
        'configuration': {'dataToExtract': 'contentAndMetadata'},
    },
}
response = requests.put(
    f'{search_endpoint}/indexers/{indexer_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=indexer, timeout=(10, 90),
)
check_response(response, (200, 201, 204))
print('Indexer ready:', indexer_name)
if indexer_exists:
    response = requests.post(
        f'{search_endpoint}/indexers/{indexer_name}/run?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    check_response(response, (202,))
    print('New run requested for the existing indexer.')


In [ ]:
for attempt in range(90):
    response = requests.get(
        f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    check_response(response, (200,))
    status = response.json()
    last_run = status.get('lastResult') or {}
    state = last_run.get('status')
    print('Run:', state, '| processed:', last_run.get('itemsProcessed'),
          '| failed:', last_run.get('itemsFailed'))
    is_requested_run = previous_run_start is None or last_run.get('startTime') != previous_run_start
    if state == 'success' and is_requested_run:
        print('Warnings:', json.dumps(last_run.get('warnings', []), indent=2))
        if last_run.get('itemsFailed', 0) or last_run.get('errors'):
            raise RuntimeError(json.dumps(last_run, indent=2))
        break
    if (is_requested_run and state in ('transientFailure', 'persistentFailure', 'error')) or status.get('status') == 'error':
        raise RuntimeError(json.dumps(last_run or status, indent=2))
    if attempt < 89:
        time.sleep(10)
else:
    raise TimeoutError('Indexer did not complete within 15 minutes. Inspect its status before retrying.')


## 7. Inspect chunks and compare search modes

Inspect chunks and their provenance, then verify stored vector lengths without printing thousands of floats. More chunks than source files demonstrates one-to-many projection.

Use the same question and `top=3` for each search. Keyword uses BM25; vector-only uses embedding similarity; hybrid combines keyword and vector ranks using reciprocal rank fusion (RRF). Semantic ranking reranks candidates and can return extractive captions; it does not generate an LLM answer. Compare filenames, chunk IDs and ordering, not raw scores across modes. With these few documents, rankings can be identical.

The text-vector queries below use the index's vectorizer, so Search's identity calls the embedding deployment. `k=50` supplies up to 50 vector candidates for semantic reranking, even though this tiny sample has fewer chunks.

In [ ]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={'search': '*', 'count': True, 'select': 'chunk_id,parent_id,chunk,source', 'top': 100},
    timeout=(10, 90),
)
check_response(response, (200,))
results = response.json()
chunks = results['value']
if not chunks:
    raise RuntimeError('No chunks were indexed. Inspect indexer errors and the Blob folder prefix.')
print('Total chunks:', results['@odata.count'])
print(json.dumps(chunks, indent=2))


### Check a sample of stored embeddings

The next cell checks that indexed chunks have vectors of the expected size. `search='*'` retrieves up to **three chunks**; this is **not a vector-similarity search**. `select` returns only each chunk's ID and stored `text_vector`.

After requiring HTTP `200`, the cell checks that every sampled vector has **1,536 values** (`embedding_dimensions`). Missing results, missing vectors or wrong-sized vectors raise an error. It prints chunk IDs and vector lengths instead of thousands of numbers.

This is a **sample check**, not verification of every indexed chunk or the quality of its embedding.

In [ ]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={'search': '*', 'select': 'chunk_id,text_vector', 'top': 3}, timeout=(10, 90),
)
check_response(response, (200,))
vector_samples = response.json()['value']
if not vector_samples or any(len(item.get('text_vector') or []) != embedding_dimensions
                             for item in vector_samples):
    raise RuntimeError('Missing or wrong-sized vectors. Reset/run the indexer and inspect embedding errors.')
for item in vector_samples:
    print(item['chunk_id'], '| vector dimensions:', len(item['text_vector']))


### Keyword search (BM25)

In [ ]:
question = 'What receipts do I need to claim travel expenses, and when must I submit them?'
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={'search': question, 'searchFields': 'chunk',
          'select': 'chunk_id,title,chunk,source', 'top': 3},
    timeout=(10, 90),
)
check_response(response, (200,))
print(json.dumps(response.json(), indent=2))


### Vector-only search

There is deliberately no `search` property: this is not a hybrid request.

**The question is embedded at query time.** With `kind='text'`, the request passes `question` to Azure AI Search. The index's configured Azure OpenAI vectorizer generates an embedding for that question, then Search compares it with the chunk vectors already stored in `text_vector`.

The indexer's embedding skill generated the document-chunk embeddings earlier; they are not regenerated for every search. Query-time embedding calls use Search's managed identity and incur embedding usage charges. The hybrid queries below also use this text-to-vector step, alongside keyword retrieval.

In [ ]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={
        'vectorQueries': [{'kind': 'text', 'text': question, 'fields': 'text_vector', 'k': 50}],
        'select': 'chunk_id,title,chunk,source', 'top': 3,
    }, timeout=(10, 90),
)
check_response(response, (200,))
print(json.dumps(response.json(), indent=2))


### Hybrid search (keyword + vector, fused with RRF)

In [ ]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={
        'search': question, 'searchFields': 'chunk',
        'vectorQueries': [{'kind': 'text', 'text': question, 'fields': 'text_vector', 'k': 50}],
        'select': 'chunk_id,title,chunk,source', 'top': 3,
    }, timeout=(10, 90),
)
check_response(response, (200,))
print(json.dumps(response.json(), indent=2))


### Keyword search + semantic ranking

Look for `@search.rerankerScore` and `@search.captions`. A partial semantic response is an error here, not a successful reranking demonstration.

In [ ]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={
        'search': question, 'searchFields': 'chunk', 'queryType': 'semantic',
        'semanticConfiguration': semantic_configuration, 'captions': 'extractive',
        'select': 'chunk_id,title,chunk,source', 'top': 3,
    }, timeout=(10, 90),
)
check_response(response, (200,))
if response.json().get('@search.semanticPartialResponseReason'):
    raise RuntimeError(f'Semantic ranking did not complete: {response.json()}')
print(json.dumps(response.json(), indent=2))


### Hybrid search + semantic ranking

Keyword and vector retrieval first produce candidates; semantic ranking reranks the fused results.

In [ ]:
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json={
        'search': question, 'searchFields': 'chunk',
        'vectorQueries': [{'kind': 'text', 'text': question, 'fields': 'text_vector', 'k': 50}],
        'queryType': 'semantic', 'semanticConfiguration': semantic_configuration,
        'captions': 'extractive', 'select': 'chunk_id,title,chunk,source', 'top': 3,
    }, timeout=(10, 90),
)
check_response(response, (200,))
if response.json().get('@search.semanticPartialResponseReason'):
    raise RuntimeError(f'Semantic ranking did not complete: {response.json()}')
print(json.dumps(response.json(), indent=2))


## References

- [Blob indexer](https://learn.microsoft.com/azure/search/search-how-to-index-azure-blob-storage)
- [Managed identity access to Storage](https://learn.microsoft.com/azure/search/search-how-to-managed-identities-storage)
- [Text Split skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-textsplit)
- [Shaper skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-shaper)
- [Azure OpenAI embedding skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-azure-openai-embedding)
- [Azure OpenAI vectorizer](https://learn.microsoft.com/azure/search/vector-search-vectorizer-azure-open-ai)
- [Hybrid search](https://learn.microsoft.com/azure/search/hybrid-search-overview)
- [Semantic ranking](https://learn.microsoft.com/azure/search/semantic-search-overview)
- [REST pipeline reference](https://github.com/memasanz/mm-ai-search/blob/main/000_CreateIndex.ipynb)
- [Index projections](https://learn.microsoft.com/azure/search/search-how-to-define-index-projections)
- [Blob deletion detection](https://learn.microsoft.com/azure/search/search-how-to-index-azure-blob-changed-deleted)

Next: [notebook 05](05-blob-content-understanding.ipynb) uses a separate Blob indexer with Content Understanding to extract and chunk a PDF with page metadata. [Notebook 06](06-push-chunks-vectors.ipynb) instead chunks TXT files and generates embeddings in Python, then pushes them to a separate index.